Environment

In [1]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [2]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [3]:
import os
import json
import pycomet
import numpy as np
import tensorflow as tf

from silx.io.dictdump import h5todict

from hypernet.apps import utils as hyputils
from pycomet.networks import NodePartitionFunction, EdgeBackward, LinearMEGraphNet

Plotting

In [4]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [5]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [6]:
# Load data
sol = h5todict(h5file='./../physics/solution.hdf5')
# Extract primitive variables
t, T = [sol[k] for k in ('t', 'T')]
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]
n = np.hstack([na, nm])
# Update thermochemistry
kin._update(n, T)

Construct graph

In [7]:
# Dimensions
nb_pts, nb_nodes = nm.shape

In [8]:
# Adjacency matrix (forward)
kf = 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
k_idx = np.where(kf > 0)

In [9]:
# Nodes
node_nm = nm[0].reshape(1,-1,1)
node_nm = np.tile(node_nm, (nb_pts,1,1))
node_Tt = np.tile(T.reshape(-1,1,1), (1,nb_nodes,1))
node_Ti = np.tile(T.reshape(-1,1,1), (1,nb_nodes,1))

In [10]:
# Edges
edge_kf = kf[k_idx].reshape(1,-1,1)
edge_kf = np.tile(edge_kf, (nb_pts,1,1))
edge_na = na[0].reshape(1,1,1)
edge_na = np.tile(edge_na, (nb_pts,1,1))

In [11]:
# Graph
graph = [node_nm, node_Tt, node_Ti, edge_kf, edge_na]
graph = tf.nest.map_structure(tf.convert_to_tensor, graph)
# Target
target = tf.convert_to_tensor(nm[...,None]) #/ const.UNA

Load linear master equation graph model

In [12]:
lays_args = {

    "node_pf": {
        "lay_id": "NodePartitionFunction",
        "kwargs": {
            "node_index_temp": {
                "int": 2,
                "chem": 1,
                "trans": 1
            },
            "specie_thermos": {
                "filename": "./data/sp_th.json",
                "thermo_process": 0,
                "eos_model": "PerfectGas"
            }
        }
    },

    "edge_bwd": {
        "lay_id": "EdgeBackward",
        "kwargs": {
            "node_attr_index_pf": 3,
            "static_fwd_edge_index": "./data/fwd_edge_index.txt"
        }
    },

    "lin_me_gnet": {
        "lay_id": "LinearMEGraphNet",
        "kwargs": {
            "node_attr_indices": [0]
        }
    }

}

In [13]:
tf.keras.backend.clear_session()

lays = {
    "node_pf": NodePartitionFunction(**lays_args["node_pf"]["kwargs"]),
    "edge_bwd": EdgeBackward(**lays_args["edge_bwd"]["kwargs"]),
    "lin_me_gnet": LinearMEGraphNet(**lays_args["lin_me_gnet"]["kwargs"])
}

def get_graph(inputs):
    node_nm, node_Tt, node_Ti, edge_kf, edge_na = inputs
    node_attr = tf.concat([node_nm,node_Tt,node_Ti], axis=-1)
    edge_attr = edge_kf*edge_na
    return (node_attr, edge_attr)

def call(inputs, training=False):
    graph = get_graph(inputs)
    # Node partition functions
    graph = lays["node_pf"](graph, training=training)
    # Edge backward
    graph = lays["edge_bwd"](graph, training=training)
    # Graph processor
    y = lays["lin_me_gnet"](graph, training=training) + inputs[0]
    return y #/ const.UNA

In [14]:
shapes = tf.nest.map_structure(lambda x: tuple(x.shape[1:]), graph)
inputs = [tf.keras.Input(shape) for shape in shapes]
model = tf.keras.Model(inputs, call(inputs))

In [15]:
def add_noise(x, stddev=0.02):
    rnd = tf.random.get_global_generator()
    dx = rnd.normal(
        shape=tf.shape(x),
        mean=0.0,
        stddev=stddev,
        dtype=x.dtype,
    )
    return x * (1 + dx)

Compute Jacobians

In [16]:
graphn = tf.nest.map_structure(tf.identity, graph)
# graphn = tf.nest.map_structure(lambda x: add_noise(x, stddev=0.2), graphn)
with tf.GradientTape(persistent=True) as g:
  g.watch(graphn[2])
  g.watch(graphn[3])
  y = model(graphn)
dy_dTi = tf.squeeze(g.batch_jacobian(y, graphn[2]))
dy_dkf = tf.squeeze(g.batch_jacobian(y, graphn[3]))

In [22]:
np.savetxt("./dy_dTi.txt", dy_dTi[-1].numpy(), fmt='%.8e')

In [ ]:
np.savetxt("./dy_dkf.txt", dy_dkf[-1].numpy(), fmt='%.8e')

Compute gradients

In [19]:
def loss(y_true, y_pred, axis=0):
    # err = tf.math.squared_difference(y_pred, y_true)
    err = tf.abs(y_pred, y_true) / y_true
    err = tf.reduce_mean(err, axis=axis)
    return tf.reduce_sum(err)

In [20]:
graphn = tf.nest.map_structure(tf.identity, graph)
# graphn = tf.nest.map_structure(lambda x: add_noise(x, stddev=0.2), graphn)
with tf.GradientTape(persistent=True) as g:
  g.watch(graphn[2])
  g.watch(graphn[3])
  y = model(graphn)
  e = loss(target, y, axis=0)
de_dTi = tf.squeeze(g.gradient(e, graphn[2]))
de_dkf = tf.squeeze(g.gradient(e, graphn[3]))

In [21]:
de_dTi

<tf.Tensor: shape=(940, 3), dtype=float64, numpy=
array([[-5.62789806e+12,  1.59874168e+13,  3.70805925e+11],
       [-3.05123341e+05,  4.39537259e+05,  7.18803044e+05],
       [-2.98852838e+05,  4.30504477e+05,  7.04031088e+05],
       ...,
       [-4.11432850e-01,  1.02484873e+00,  2.62482689e-01],
       [-4.11432850e-01,  1.02484873e+00,  2.62482689e-01],
       [-4.11432850e-01,  1.02484873e+00,  2.62482689e-01]])>